## import 

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

df.columns = [c.strip() for c in df.columns]
if "Timestamp" in df.columns:
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.sort_values("Timestamp").drop_duplicates(subset=["Timestamp"]).reset_index(drop=True)

df.head()

,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,WD,WDstdev,BP,Cleaning,Precipitation,TModA,TModB,Comments
0,2021-10-30 00:01:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.1,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
1,2021-10-30 00:02:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.2,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
2,2021-10-30 00:03:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.2,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
3,2021-10-30 00:04:00,-0.7,0.0,-0.8,0.0,0.0,21.9,99.3,0.0,0.0,0.0,0.0,0.0,1002,0,0.1,22.3,22.6,NaN
4,2021-10-30 00:05:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.3,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN


## path

In [3]:
data_path = '../data/sierraleone-bumbuna.csv'
df = pd.read_csv(data_path)


## data cleaning

In [6]:
display(df.describe(include="all"))
nulls = df.isna().mean().sort_values(ascending=False)
display(nulls)
over5 = nulls[nulls > 0.05]
print("Columns with >5% nulls:", list(over5.index))


,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,WD,WDstdev,BP,Cleaning,Precipitation,TModA,TModB,Comments
count,525600,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,0.0
mean,2022-04-30 12:00:30.000000768,201.957515,116.376337,113.720571,206.643095,198.114691,26.319394,79.448857,1.146113,1.691606,0.363823,133.044668,7.172220,999.876469,0.000967,0.004806,32.504263,32.593091,NaN
min,2021-10-30 00:01:00,-19.500000,-7.800000,-17.900000,0.000000,0.000000,12.300000,9.900000,0.000000,0.000000,0.000000,0.000000,0.000000,993.000000,0.000000,0.000000,10.700000,11.100000,NaN
25%,2022-01-29 06:00:45,-2.800000,-0.300000,-3.800000,0.000000,0.000000,23.100000,68.700000,0.000000,0.000000,0.000000,0.000000,0.000000,999.000000,0.000000,0.000000,23.500000,23.800000,NaN
50%,2022-04-30 12:00:30,0.300000,-0.100000,-0.100000,3.600000,3.400000,25.300000,85.400000,0.800000,1.600000,0.400000,161.500000,6.200000,1000.000000,0.000000,0.000000,26.600000,26.900000,NaN
75%,2022-07-30 18:00:15,362.400000,107.000000,224.700000,359.500000,345.400000,29.400000,96.700000,2.000000,2.600000,0.600000,234.100000,12.000000,1001.000000,0.000000,0.000000,40.900000,41.300000,NaN
max,2022-10-30 00:00:00,1499.000000,946.000000,892.000000,1507.000000,1473.000000,39.900000,100.000000,19.200000,23.900000,4.100000,360.000000,98.400000,1006.000000,1.000000,2.400000,72.800000,70.400000,NaN
std,NaN,298.495150,218.652659,158.946032,300.896893,288.889073,4.398605,20.520775,1.239248,1.617053,0.295000,114.284792,7.535093,2.104419,0.031074,0.047556,12.434899,12.009161,NaN


Comments         1.0
GHI              0.0
Timestamp        0.0
DNI              0.0
DHI              0.0
ModB             0.0
ModA             0.0
RH               0.0
WS               0.0
WSgust           0.0
Tamb             0.0
WSstdev          0.0
WD               0.0
BP               0.0
WDstdev          0.0
Cleaning         0.0
Precipitation    0.0
TModA            0.0
TModB            0.0
dtype: float64

Columns with >5% nulls: ['Comments']


## Solar plausibilty rules

In [7]:

dfc = df.copy()

# Irradiance can't be negative
for c in ["GHI","DNI","DHI","ModA","ModB"]:
    if c in dfc: dfc.loc[dfc[c] < 0, c] = np.nan

# Physical relationship: DHI <= GHI when both exist
if {"DHI","GHI"}.issubset(dfc.columns):
    dfc.loc[dfc["DHI"] > dfc["GHI"], "DHI"] = np.nan

# RH in [0,100]
if "RH" in dfc: dfc.loc[(dfc["RH"] < 0) | (dfc["RH"] > 100), "RH"] = np.nan

# Wind speeds >= 0
for c in ["WS","WSgust","WSstdev"]:
    if c in dfc: dfc.loc[dfc[c] < 0, c] = np.nan

# Wind direction in [0,360]
if "WD" in dfc: dfc.loc[(dfc["WD"] < 0) | (dfc["WD"] > 360), "WD"] = np.nan
if "WDstdev" in dfc: dfc.loc[dfc["WDstdev"] < 0, "WDstdev"] = np.nan

# BP (pressure) plausible sea-level range
if "BP" in dfc: dfc.loc[(dfc["BP"] < 800) | (dfc["BP"] > 1100), "BP"] = np.nan

# Precip >= 0
if "Precipitation" in dfc: dfc.loc[dfc["Precipitation"] < 0, "Precipitation"] = np.nan

# Temperatures in a broad real-world range
for c in ["Tamb","TModA","TModB"]:
    if c in dfc: dfc.loc[(dfc[c] < -50) | (dfc[c] > 80), c] = np.nan

df_after_rules = dfc.copy()
df_after_rules.describe().T.head(10)


,count,mean,min,25%,50%,75%,max,std
Timestamp,525600,2022-04-30 12:00:30.000000768,2021-10-30 00:01:00,2022-01-29 06:00:45,2022-04-30 12:00:30,2022-07-30 18:00:15,2022-10-30 00:00:00,NaN
GHI,264465.0,406.239777,0.0,132.8,359.5,654.4,1499.0,305.063927
DNI,259248.0,236.240484,0.0,1.2,115.3,461.5,946.0,261.87106
DHI,249778.0,238.971466,0.0,123.2,231.5,342.1,892.0,148.168477
ModA,525600.0,206.643095,0.0,0.0,3.6,359.5,1507.0,300.896893
ModB,525600.0,198.114691,0.0,0.0,3.4,345.4,1473.0,288.889073
Tamb,525600.0,26.319394,12.3,23.1,25.3,29.4,39.9,4.398605
RH,525600.0,79.448857,9.9,68.7,85.4,96.7,100.0,20.520775
WS,525600.0,1.146113,0.0,0.0,0.8,2.0,19.2,1.239248
WSgust,525600.0,1.691606,0.0,0.0,1.6,2.6,23.9,1.617053


## Z- Score outlier

In [8]:
z_cols = [c for c in ["GHI","DNI","DHI","ModA","ModB","WS","WSgust"] if c in dfc.columns]
for c in z_cols:
    s = pd.to_numeric(dfc[c], errors="coerce")
    mask = s.notna()
    if mask.sum() > 0:
        z = stats.zscore(s[mask], nan_policy="omit")
        out_mask = pd.Series(False, index=dfc.index)
        out_mask[mask.index[mask]] = np.abs(z) > 3
        dfc.loc[out_mask, c] = np.nan

df_after_outliers = dfc.copy()
dfc[z_cols].describe()


,GHI,DNI,DHI,ModA,ModB,WS,WSgust
count,264378.000000,259248.000000,249411.000000,523996.000000,523559.000000,521633.000000,521935.000000
mean,405.921735,236.240484,238.266228,203.665472,194.461795,1.109028,1.645261
std,304.608925,261.871060,147.126567,296.477652,283.426275,1.161456,1.514638
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,132.700000,1.200000,123.000000,0.000000,0.000000,0.000000,0.000000
50%,359.400000,115.300000,231.200000,3.200000,2.900000,0.800000,1.600000
75%,654.100000,461.500000,341.700000,355.200000,340.300000,1.900000,2.600000
max,1320.000000,946.000000,683.400000,1109.200000,1064.000000,4.800000,6.400000


## Medain Imputation

In [9]:
num_cols = dfc.select_dtypes(include=[np.number]).columns
for c in num_cols:
    med = dfc[c].median()
    dfc[c] = dfc[c].fillna(med)

df_clean = dfc.copy()
df_clean.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525600 entries, 0 to 525599
Data columns (total 19 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Timestamp      525600 non-null  datetime64[ns]
 1   GHI            525600 non-null  float64       
 2   DNI            525600 non-null  float64       
 3   DHI            525600 non-null  float64       
 4   ModA           525600 non-null  float64       
 5   ModB           525600 non-null  float64       
 6   Tamb           525600 non-null  float64       
 7   RH             525600 non-null  float64       
 8   WS             525600 non-null  float64       
 9   WSgust         525600 non-null  float64       
 10  WSstdev        525600 non-null  float64       
 11  WD             525600 non-null  float64       
 12  WDstdev        525600 non-null  float64       
 13  BP             525600 non-null  float64       
 14  Cleaning       525600 non-null  int64         
 15  

c:\Users\user\Downloads\Solar-challenge-week0\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## quick profiling and missingness check point

In [10]:
display(df.describe(include="all").T.round(3))                     # raw
display(df_after_rules[z_cols].describe().T.round(3))              # after solar rules
display(df_after_outliers[z_cols].describe().T.round(3))           # after outliers
display(df_clean.describe(include="all").T.round(3))               # final
missing = df_clean.isna().mean().sort_values(ascending=False)
missing.head(12)


,count,mean,min,25%,50%,75%,max,std
Timestamp,525600,2022-04-30 12:00:30.000000768,2021-10-30 00:01:00,2022-01-29 06:00:45,2022-04-30 12:00:30,2022-07-30 18:00:15,2022-10-30 00:00:00,NaN
GHI,525600.0,201.957515,-19.5,-2.8,0.3,362.4,1499.0,298.49515
DNI,525600.0,116.376337,-7.8,-0.3,-0.1,107.0,946.0,218.652659
DHI,525600.0,113.720571,-17.9,-3.8,-0.1,224.7,892.0,158.946032
ModA,525600.0,206.643095,0.0,0.0,3.6,359.5,1507.0,300.896893
ModB,525600.0,198.114691,0.0,0.0,3.4,345.4,1473.0,288.889073
Tamb,525600.0,26.319394,12.3,23.1,25.3,29.4,39.9,4.398605
RH,525600.0,79.448857,9.9,68.7,85.4,96.7,100.0,20.520775
WS,525600.0,1.146113,0.0,0.0,0.8,2.0,19.2,1.239248
WSgust,525600.0,1.691606,0.0,0.0,1.6,2.6,23.9,1.617053


,count,mean,std,min,25%,50%,75%,max
GHI,264465.0,406.240,305.064,0.0,132.8,359.5,654.4,1499.0
DNI,259248.0,236.240,261.871,0.0,1.2,115.3,461.5,946.0
DHI,249778.0,238.971,148.168,0.0,123.2,231.5,342.1,892.0
ModA,525600.0,206.643,300.897,0.0,0.0,3.6,359.5,1507.0
ModB,525600.0,198.115,288.889,0.0,0.0,3.4,345.4,1473.0
WS,525600.0,1.146,1.239,0.0,0.0,0.8,2.0,19.2
WSgust,525600.0,1.692,1.617,0.0,0.0,1.6,2.6,23.9


,count,mean,std,min,25%,50%,75%,max
GHI,264378.0,405.922,304.609,0.0,132.7,359.4,654.1,1320.0
DNI,259248.0,236.240,261.871,0.0,1.2,115.3,461.5,946.0
DHI,249411.0,238.266,147.127,0.0,123.0,231.2,341.7,683.4
ModA,523996.0,203.665,296.478,0.0,0.0,3.2,355.2,1109.2
ModB,523559.0,194.462,283.426,0.0,0.0,2.9,340.3,1064.0
WS,521633.0,1.109,1.161,0.0,0.0,0.8,1.9,4.8
WSgust,521935.0,1.645,1.515,0.0,0.0,1.6,2.6,6.4


,count,mean,min,25%,50%,75%,max,std
Timestamp,525600,2022-04-30 12:00:30.000000768,2021-10-30 00:01:00,2022-01-29 06:00:45,2022-04-30 12:00:30,2022-07-30 18:00:15,2022-10-30 00:00:00,NaN
GHI,525600.0,382.800539,0.0,356.7,359.4,362.2,1320.0,217.28514
DNI,525600.0,174.952927,0.0,115.3,115.3,115.3,946.0,193.599354
DHI,525600.0,234.553111,0.0,231.2,231.2,231.2,683.4,101.410708
ModA,525600.0,203.053701,0.0,0.0,3.2,353.8,1109.2,296.231357
ModB,525600.0,193.717925,0.0,0.0,2.9,338.5,1064.0,283.126225
Tamb,525600.0,26.319394,12.3,23.1,25.3,29.4,39.9,4.398605
RH,525600.0,79.448857,9.9,68.7,85.4,96.7,100.0,20.520775
WS,525600.0,1.106696,0.0,0.0,0.8,1.9,4.8,1.157374
WSgust,525600.0,1.644945,0.0,0.0,1.6,2.6,6.4,1.509352


Comments     1.0
GHI          0.0
Timestamp    0.0
DNI          0.0
DHI          0.0
ModB         0.0
ModA         0.0
RH           0.0
WS           0.0
WSgust       0.0
Tamb         0.0
WSstdev      0.0
dtype: float64

## Time - series

In [ ]:
if "Timestamp" in df_clean:
    df_clean["Month"] = df_clean["Timestamp"].dt.month
    df_clean["Hour"]  = df_clean["Timestamp"].dt.hour

    for col in ["GHI","DNI","DHI","Tamb"]:
        if col in df_clean.columns:
            plt.figure(figsize=(10,3))
            plt.plot(df_clean["Timestamp"], df_clean[col])
            plt.title(f"{col} over time")
            plt.xlabel("Timestamp"); plt.ylabel(col)
            plt.tight_layout(); plt.show()

    if "GHI" in df_clean.columns:
        plt.figure()
        df_clean.groupby("Month")["GHI"].mean().plot(kind="bar", title="Average GHI by Month")
        plt.tight_layout(); plt.show()


 ## cleaning and impact

In [ ]:
if "Cleaning" in df_clean.columns and {"ModA","ModB"}.issubset(df_clean.columns):
    avg = df_clean.groupby("Cleaning")[["ModA","ModB"]].mean()
    display(avg)
    for col in ["ModA","ModB"]:
        plt.figure()
        avg[col].plot(kind="bar", title=f"{col} by Cleaning (0/1)")
        plt.tight_layout(); plt.show()


## correlation

In [ ]:
corr_cols = [c for c in ["GHI","DNI","DHI","TModA","TModB","Tamb","RH","WS","WSgust"] if c in df_clean.columns]
if len(corr_cols) >= 2:
    cmat = df_clean[corr_cols].corr()
    display(cmat.round(3))
    plt.figure(figsize=(6,5))
    plt.imshow(cmat, aspect="auto")
    plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
    plt.yticks(range(len(corr_cols)), corr_cols)
    plt.colorbar(); plt.title("Correlation Heatmap")
    plt.tight_layout(); plt.show()

if {"WS","GHI"}.issubset(df_clean.columns):
    plt.figure(); plt.scatter(df_clean["WS"], df_clean["GHI"], alpha=0.35)
    plt.xlabel("WS"); plt.ylabel("GHI"); plt.title("WS vs GHI")
    plt.tight_layout(); plt.show()

if {"RH","Tamb"}.issubset(df_clean.columns):
    plt.figure(); plt.scatter(df_clean["RH"], df_clean["Tamb"], alpha=0.35)
    plt.xlabel("RH"); plt.ylabel("Tamb"); plt.title("RH vs Tamb")
    plt.tight_layout(); plt.show()


## wind and distribution

# Histograms
for col in ["GHI","WS"]:
    if col in df.columns:
        plt.figure(); plt.hist(df[col].dropna(), bins=40)
        plt.title(f"{col} distribution"); plt.tight_layout(); plt.show()

# Wind rose is optional (needs 'windrose' package); skip in CI, do locally if installed
try:
    from windrose import WindroseAxes
    if {"WS","WD"}.issubset(df.columns):
        ax = WindroseAxes.from_ax()
        ax.bar(df["WD"], df["WS"], normed=True, opening=0.8)
        ax.set_title("Wind Rose (WS by WD)")
        plt.show()
except Exception as e:
    print("Windrose not available or data missing; skipping. Reason:", e)


## Temprature / RH influence

In [ ]:
# Simple check of RH influence on Tamb (scatter w/ alpha)
if {"RH","Tamb"}.issubset(df.columns):
    plt.figure(); plt.scatter(df["RH"], df["Tamb"], alpha=0.3)
    plt.xlabel("RH"); plt.ylabel("Tamb"); plt.title("RH vs Tamb")
    plt.tight_layout(); plt.show()

# Bubble: GHI vs Tamb with bubble size = RH (or BP)
size_col = "RH" if "RH" in df.columns else ("BP" if "BP" in df.columns else None)
if size_col and {"GHI","Tamb"}.issubset(df.columns):
    s = df[size_col].astype(float)
    s = (s - s.min()) / (s.max() - s.min() + 1e-9) * 400 + 10
    plt.figure(); plt.scatter(df["Tamb"], df["GHI"], s=s, alpha=0.3)
    plt.xlabel("Tamb"); plt.ylabel("GHI"); plt.title(f"GHI vs Tamb (bubble={size_col})")
    plt.tight_layout(); plt.show()


## Bubble chart

In [ ]:
size_col = "RH" if "RH" in df_clean.columns else ("BP" if "BP" in df_clean.columns else None)
if size_col and {"GHI","Tamb"}.issubset(df_clean.columns):
    s = df_clean[size_col].astype(float)
    s = (s - s.min()) / (s.max() - s.min() + 1e-9) * 400 + 10
    plt.figure()
    plt.scatter(df_clean["Tamb"], df_clean["GHI"], s=s, alpha=0.3)
    plt.xlabel("Tamb (°C)"); plt.ylabel("GHI (W/m²)")
    plt.title(f"GHI vs Tamb (bubble={size_col})")
    plt.tight_layout(); plt.show()


## Exporting cleaned csv

In [ ]:
Path("data").mkdir(exist_ok=True)
out = Path("data") / "benin_clean.csv"
df_clean.to_csv(out, index=False)
print(f"[OK] saved: {out}")
if "Timestamp" in df_clean:
    print("Timestamp dtype:", df_clean["Timestamp"].dtype)
print("Rows:", len(df_clean), "Cols:", len(df_clean.columns))
